### Functions

In [1]:
def load_session_data(subject, date):
    """Load all data for a given subject and date"""
    import sys
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig
    
    
    # Load session
    loader = NeuralDataLoader()
    loader.load_session(subject, date)
    config = Dots3DMPConfig(subject)

    # spike data (unit, trial, time)
    stimOn_spikes = loader.get_spike_data(alignment='stimOn', good_units_only=True, good_trials_only=True)
    saccOnset_spikes = loader.get_spike_data(alignment='saccOnset', good_units_only=True, good_trials_only=True)
    postTargHold_spikes = loader.get_spike_data(alignment='postTargHold', good_units_only=True, good_trials_only=True)
    tuning_spikes = loader.get_tuning_data(good_units_only=True, good_trials_only=True)

    # behavioral data
    behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True, cal_mean_RT=True)
    behavior_tuning = loader.get_behavioral_data(task='tuning', good_trials_only=True)
    behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
    behavior_tuning_converted = config.convert_behavioral_data(behavior_tuning, task='tuning')

    # Unit Info
    unit_info = loader.get_unit_info(good_units_only=True)
    MST_units = loader.get_units_by_area(unit_info, area_name='MST')
    VPS_units = loader.get_units_by_area(unit_info, area_name='VPS')
    dual_units = loader.get_units_by_area(unit_info, area_name='dual')

    # Time Info
    time_info = config.get_time_Info('dots3DMP')
    time_info_tuning = config.get_time_Info('tuning')
    time_axes_dots3DMP = config.get_time_axes('dots3DMP')
    time_axes_tuning = config.get_time_axes('tuning')

    # Prepare data
    spikes_data = {
        'stimOn': stimOn_spikes,
        'saccOnset': saccOnset_spikes,
        'postTargHold': postTargHold_spikes,
    }

    tuning_spikes_data = {'stimOn': tuning_spikes}
    
    units_data = {
        'MST': MST_units,
        'VPS': VPS_units,
        'dual': dual_units
    }
    
    return {
        'loader': loader,
        'config': config,
        'spikes_data': spikes_data,
        'tuning_spikes_data': tuning_spikes_data,
        'behavior_converted': behavior_converted,
        'behavior_tuning_converted': behavior_tuning_converted,
        'unit_info': unit_info,
        'units_data': units_data,
        'time_axes_dots3DMP': time_axes_dots3DMP,
        'time_axes_tuning': time_axes_tuning,  
        'time_info': time_info,
        'time_info_tuning': time_info_tuning,
    }

In [3]:
# data = load_session_data('zarya', '20250710')
data[ 'behavior_converted']['mean_RT']

{'mod1_coh1': 0.5946132830211097,
 'mod2_coh1': 0.8925021693820047,
 'mod2_coh2': 0.7702658664612544,
 'mod3_coh1': 0.7347089818183412,
 'mod3_coh2': 0.6973434488824074}

In [10]:
def pool_all_sessions(session_list, alignment='stimOn', save_pooled=True, is_RT=False):
    import sys
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_single_neurons.neural_feature_analyzer import NeuralFeatureAnalyzer
    import pandas as pd
    import os

    all_features = []
    failed_sessions = []
    
    # Set save directory and task type based on alignment and RT flag
    if alignment == 'stimOn':
        save_dir = r'D:\Neural-Pipeline\results\analysis_single_neurons\dots3DMPtuning_neuralfeatures'
        if is_RT:
            task_type = 'tuning_RT'
            print("Task type: TUNING with RT analysis")
        else:
            task_type = 'tuning'
            print("Task type: TUNING (standard)")
    else:  # saccOnset
        save_dir = r'D:\Neural-Pipeline\results\analysis_single_neurons\dots3DMP_neuralfeatures'
        task_type = 'regular'
        print("Task type: REGULAR")
    
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"Processing {len(session_list)} sessions with alignment '{alignment}' ({task_type} task)...")
    print("=" * 60)
    
    for i, (subject, date, abs_pos, rel_pos) in enumerate(session_list):
        try:
            print(f"\nProcessing session {i+1}/{len(session_list)}: {subject} {date}")
            print(f"  Position: absolute {abs_pos}, relative {rel_pos}")
            print(f"  Task type: {task_type} (alignment: {alignment})")
            
            # Load and analyze session
            session_data = load_session_data(subject, date)
            if is_RT:
                mean_RT = session_data['behavior_converted']['mean_RT']
            else:
                mean_RT = None

            analyzer = NeuralFeatureAnalyzer(session_data, subject, date, 
                                           alignment=alignment,  # Pass alignment parameter
                                           session_position=abs_pos, 
                                           relative_position=rel_pos, mean_RT = mean_RT)
            
            # Analyze all units (alignment is already set in constructor)
            analyzer.analyze_all_units()
            
            # Get features
            session_features = analyzer.get_features_dataframe()
            
            if len(session_features) > 0:
                all_features.append(session_features)
                
                # Quick summary
                n_MST = len(session_features[session_features['area'] == 'MST'])
                n_VPS = len(session_features[session_features['area'] == 'VPS'])
                n_dual = len(session_features[session_features['area'] == 'dual'])
                n_unknown = len(session_features[session_features['area'] == 'unknown'])
                mean_rate = session_features['overall_firing_rate'].mean()
                
                print(f"  ✓ Added {len(session_features)} units")
                print(f"    MST: {n_MST}, VPS: {n_VPS}, dual: {n_dual}, unknown: {n_unknown}")
                print(f"    Mean firing rate: {mean_rate:.2f} Hz")
                
                # Show tuning-specific info if applicable
                if alignment == 'stimOn' and 'tuning_n_windows' in session_features.columns:
                    mean_windows = session_features['tuning_n_windows'].mean()
                    print(f"    Mean time windows analyzed: {mean_windows:.1f}")
                
            else:
                print(f"  ⚠ No units found in session")
            
        except Exception as e:
            print(f"  ✗ Failed to process {subject} {date}: {e}")
            failed_sessions.append((subject, date))
            continue
    
    # Combine and save results
    if all_features:
        pooled_features = pd.concat(all_features, ignore_index=True)
        
        print("\n" + "=" * 60)
        print("POOLING SUMMARY")
        print("=" * 60)
        print(f"Successfully pooled {len(pooled_features)} units from {len(all_features)}/{len(session_list)} sessions")
        print(f"Task type: {task_type} (alignment: {alignment})")
        
        if failed_sessions:
            print(f"Failed sessions: {[f'{s}_{d}' for s, d in failed_sessions]}")
        
        if save_pooled:
            # Create filename based on alignment and task type
            pooled_filename = f"zarya_pooled_neural_features_{task_type}_{alignment}.csv"
            pooled_path = os.path.join(save_dir, pooled_filename)
            pooled_features.to_csv(pooled_path, index=False)
            print(f"\nDataset saved to: {pooled_path}")
        
        # Final summary
        area_counts = pooled_features['area'].value_counts()
        print(f"\nFinal dataset: {len(pooled_features)} units")
        for area, count in area_counts.items():
            print(f"  {area}: {count} units ({count/len(pooled_features)*100:.1f}%)")
        
        # Show additional stats for tuning data
        if alignment == 'stimOn':
            print(f"\nTuning-specific statistics:")
            if 'tuning_n_windows' in pooled_features.columns:
                print(f"  Average time windows per unit: {pooled_features['tuning_n_windows'].mean():.1f}")
            
            # Show neurometric threshold stats
            for modality in ['ves', 'vis', 'comb']:
                threshold_col = f'{modality}_neurometric_threshold'
                if threshold_col in pooled_features.columns:
                    valid_thresholds = pooled_features[threshold_col].dropna()
                    if len(valid_thresholds) > 0:
                        print(f"  {modality.upper()} neurometric thresholds: {len(valid_thresholds)} units, "
                              f"mean = {valid_thresholds.mean():.2f}°")
        
        return pooled_features
    else:
        print("❌ No sessions were successfully processed!")
        return pd.DataFrame()

# Usage examples:
# For tuning data (default):
# pooled_tuning = pool_all_sessions(session_list, alignment='stimOn')

# For regular task data:
# pooled_regular = pool_all_sessions(session_list, alignment='saccOnset')

### Main code

In [11]:
# Define Zarya sessions
subject = 'zarya'
dates = ['20250417', '20250501', '20250523', '20250602', '20250702', '20250710']
center = (4, 4)

# Position for each session (one position per session)
positions = [
    (4, 3),  # 20250417
    (5, 4),  # 20250501
    (4, 4),  # 20250523
    (4, 5),  # 20250602
    (4, 3),  # 20250702
    (5, 4)   # 20250710
]

# Calculate relative positions from center
relative_positions = [(pos[0] - center[0], pos[1] - center[1]) for pos in positions]

# Create session list with positions
session_list = [(subject, dates[i], positions[i], relative_positions[i]) for i in range(len(dates))]

# # # # Process tuning data (default behavior)
print("Processing TUNING data...")
pooled_tuning_data = pool_all_sessions(session_list, alignment='stimOn')

# # Process regular task data
print("\n" + "="*80)
print("Processing REGULAR TASK data...")
pooled_regular_data = pool_all_sessions(session_list, alignment='saccOnset')

# Process tuning task_RT
print("\n" + "="*80)
print("Processing TUNING_RT data...")
pool_all_sessions(session_list, alignment='stimOn', save_pooled=True, is_RT = True)

# Optional: Display summary comparison
# print("\n" + "="*80)
# print("COMPARISON SUMMARY")
# print("="*80)
# print(f"Tuning data: {len(pooled_tuning_data)} units")
# print(f"Regular task data: {len(pooled_regular_data)} units")

# if len(pooled_tuning_data) > 0:
#     print(f"\nTuning data areas:")
#     tuning_areas = pooled_tuning_data['area'].value_counts()
#     for area, count in tuning_areas.items():
#         print(f"  {area}: {count}")

if len(pooled_regular_data) > 0:
    print(f"\nRegular task data areas:")
    regular_areas = pooled_regular_data['area'].value_counts()
    for area, count in regular_areas.items():
        print(f"  {area}: {count}")


Processing TUNING data...
Task type: TUNING (standard)
Processing 6 sessions with alignment 'stimOn' (tuning task)...

Processing session 1/6: zarya 20250417
  Position: absolute (4, 3), relative (0, -1)
  Task type: tuning (alignment: stimOn)
Loaded dots3DMP data: zarya20250417dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250417dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya
Debug: Session 20250417 at position (4, 3) (relative: (0, -1))
Debug: Alignment: stimOn, Is tuning: True
Debug: Number of units determined: 176
Analyzing units for zarya 20250417 with alignment stimOn
Processing 176 units...
  Processing unit 1/176
  Processing unit 2/176
  Processing unit 3/176
  Processing unit 4/176
  Processing unit 5/176
  Processing unit 11/176
  Processing unit 21/176
  Processing unit 31/176
  Processing unit 41/176
  Processing unit 51/176
  Processing unit 61/176
  Processing unit 71/176
  Processing unit 81/176
  Processing unit 91/176
  Process